# Installations, imports, utils

In [1]:
!pip install transformers==4.33.0 accelerate==0.22.0 einops==0.6.1 langchain==0.0.300 xformers==0.0.21 \
bitsandbytes==0.41.1 sentence_transformers==2.2.2 chromadb==0.4.12 --quiet

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-cloud-pubsublite 1.8.2 requires overrides<7.0.0,>=6.0.1, but you have overrides 7.7.0 which is incompatible.
jupyterlab-lsp 4.2.0 requires jupyter-lsp>=2.0.0, but you have jupyter-lsp 1.5.1 which is incompatible.
torchdata 0.6.0 requires torch==2.0.0, but you have torch 2.0.1 which is incompatible.


In [2]:
!pip install rouge-score==0.0.4 nltk==3.8.1 bert-score==0.3.11 --quiet

import numpy as np
import torch
from sklearn.metrics import precision_score, recall_score
from nltk.translate.bleu_score import sentence_bleu
from rouge_score import rouge_scorer
from bert_score import score as bert_score
from transformers import GPT2LMHeadModel, GPT2Tokenizer

import nltk
nltk.download('wordnet')
nltk.download('omw-1.4')
from nltk.translate.meteor_score import meteor_score

from transformers import GPT2LMHeadModel, GPT2Tokenizer

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
preprocessing 0.1.13 requires nltk==3.2.4, but you have nltk 3.8.1 which is incompatible.


/opt/conda/lib/python3.10/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.23.5
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"
[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /usr/share/nltk_data...


In [3]:
from torch import cuda, bfloat16
import torch
import transformers
from transformers import AutoTokenizer
from time import time

from langchain.llms import HuggingFacePipeline
from langchain.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.chains import RetrievalQA
from langchain.vectorstores import Chroma


# Initialize model, tokenizer, query pipeline

Define the model, the device, and the `bitsandbytes` configuration.

In [4]:
model_id = '/kaggle/input/llama-2/pytorch/7b-chat-hf/1'

device = f'cuda:{cuda.current_device()}' if cuda.is_available() else 'cpu'

# this requires the `bitsandbytes` library
bnb_config = transformers.BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=bfloat16
)

Prepare the model and the tokenizer.

In [5]:
time_1 = time()
model_config = transformers.AutoConfig.from_pretrained(
    model_id,
)
model = transformers.AutoModelForCausalLM.from_pretrained(
    model_id,
    trust_remote_code=True,
    config=model_config,
    quantization_config=bnb_config,
    device_map='auto',
)
tokenizer = AutoTokenizer.from_pretrained(model_id)
time_2 = time()
print(f"Prepare model, tokenizer: {round(time_2-time_1, 3)} sec.")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Prepare model, tokenizer: 185.604 sec.


/opt/conda/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:362: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`. This was detected when initializing the generation config instance, which means the corresponding file may hold incorrect parameterization and should be fixed.
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:367: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`. This was detected when initializing the generation config instance, which means the corresponding file may hold incorrect parameterization and should be fixed.
  warnings.warn(


Define the query pipeline.

In [6]:
time_1 = time()
query_pipeline = transformers.pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        torch_dtype=torch.float16,
        device_map="auto",)
time_2 = time()
print(f"Prepare pipeline: {round(time_2-time_1, 3)} sec.")

Prepare pipeline: 1.484 sec.


We define a function for testing the pipeline.

In [7]:
def test_model(tokenizer, pipeline, prompt_to_test):
    """
    Perform a query
    print the result
    Args:
        tokenizer: the tokenizer
        pipeline: the pipeline
        prompt_to_test: the prompt
    Returns
        None
    """
    # adapted from https://huggingface.co/blog/llama2#using-transformers
    time_1 = time()
    sequences = pipeline(
        prompt_to_test,
        do_sample=True,
        top_k=10,
        num_return_sequences=1,
        eos_token_id=tokenizer.eos_token_id,
        max_length=2048,)
    time_2 = time()
    print(f"Test inference: {round(time_2-time_1, 3)} sec.")
    for seq in sequences:
        print(f"Result: {seq['generated_text']}")

## Test the query pipeline

We test the pipeline with a query about the meaning of State of the Union (SOTU).

In [8]:
test_model(tokenizer,
           query_pipeline,
           "Please explain what is the State of the Union address. Give just a definition. Keep it in 100 words.")

/opt/conda/lib/python3.10/site-packages/transformers/generation/utils.py:1417: UserWarning: You have modified the pretrained model configuration to control generation. This is a deprecated strategy to control generation and will be removed soon, in a future version. Please use a generation configuration file (see https://huggingface.co/docs/transformers/main_classes/text_generation )
  warnings.warn(


Test inference: 8.166 sec.
Result: Please explain what is the State of the Union address. Give just a definition. Keep it in 100 words.
The State of the Union address is an annual speech delivered by the President of the United States to Congress, in which they report on the current state of the nation, highlight their accomplishments, and propose new policies and initiatives. The address has been a tradition since 1790 and is seen as an important moment for the President to set the legislative agenda and rally the country around a common vision.


In [137]:
import pandas as pd

data = pd.read_csv('/kaggle/input/glassdoor-company-insightsscraped-data-collection/glassdoor_comany.csv', encoding='latin1')
data

,Company Name,Company rating,Company reviews,Company salaries,Company Jobs,Location,Number of Employees,Industry Type,Company Description
0,Amazon,3.8,168.8K,201.7K,201.7K,25 office locations in United States,10000+ Employees,Internet & Web Services,"All Amazon teams and businesses, from Prime de..."
1,Deloitte,4.1,97.2K,167.4K,167.4K,23 office locations in United States,10000+ Employees,Accounting & Tax,Think a professional services career is nothin...
2,Target,3.6,75K,116.1K,116.1K,4 office locations in United States,10000+ Employees,General Merchandise & Superstores,Target is one of the worlds most recognized b...
3,McDonald's,3.5,117K,71.7K,71.7K,110 N Carpenter Street,10000+ Employees,Restaurants & Cafes,McDonalds is proud to be one of the most reco...
4,Accenture,4.0,145.1K,53.6K,53.6K,United States,10000+ Employees,Business Consulting,Accenture is a global professional services co...
...,...,...,...,...,...,...,...,...,...
9935,Furman University,4.0,169,229,229,United States,501 to 1000 Employees,Colleges & Universities,"This school's slogan could be, ""Go Further tha..."
9936,Kilwin's Chocolates,3.8,186,192,192,United States,51 to 200 Employees,Food & Beverage Manufacturing,TOA Global is the market leader in dedicated o...
9937,TOA Global,4.3,618,21,21,United States,1001 to 5000 Employees,HR Consulting,Insurance broking and risk management solution...
9938,JLT Group,3.7,543,51,51,United States,5001 to 10000 Employees,Insurance Agencies & Brokerages,A steakhouse experience should be unforgettabl...


In [138]:
data= data[:200]
data

,Company Name,Company rating,Company reviews,Company salaries,Company Jobs,Location,Number of Employees,Industry Type,Company Description
0,Amazon,3.8,168.8K,201.7K,201.7K,25 office locations in United States,10000+ Employees,Internet & Web Services,"All Amazon teams and businesses, from Prime de..."
1,Deloitte,4.1,97.2K,167.4K,167.4K,23 office locations in United States,10000+ Employees,Accounting & Tax,Think a professional services career is nothin...
2,Target,3.6,75K,116.1K,116.1K,4 office locations in United States,10000+ Employees,General Merchandise & Superstores,Target is one of the worlds most recognized b...
3,McDonald's,3.5,117K,71.7K,71.7K,110 N Carpenter Street,10000+ Employees,Restaurants & Cafes,McDonalds is proud to be one of the most reco...
4,Accenture,4.0,145.1K,53.6K,53.6K,United States,10000+ Employees,Business Consulting,Accenture is a global professional services co...
...,...,...,...,...,...,...,...,...,...
195,Randstad US,3.7,5.1K,11.1K,11.1K,25 office locations in United States,5001 to 10000 Employees,HR Consulting,Randstad is the global leader in the HR servic...
196,Adecco,3.7,8.7K,5K,5K,"10151 Deerwood Park Blvd, Bldg 400",201 to 500 Employees,Staffing & Subcontracting,"eBay is where the world goes to shop, sell, an..."
197,eBay,4.1,5.4K,9.4K,9.4K,15 office locations in United States,10000+ Employees,Internet & Web Services,UST is a global digital transformations soluti...
198,UST,3.9,7.2K,9.6K,9.6K,8 office locations in United States,10000+ Employees,Information Technology Support Services,Genpact (NYSE: G) is a global professional ser...


In [140]:
data.drop(['Company reviews', 'Company salaries', 'Company Jobs', 'Number of Employees'], axis=1, inplace=True)
data

/tmp/ipykernel_32/154685627.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data.drop(['Company reviews', 'Company salaries', 'Company Jobs', 'Number of Employees'], axis=1, inplace=True)


,Company Name,Company rating,Location,Industry Type,Company Description
0,Amazon,3.8,25 office locations in United States,Internet & Web Services,"All Amazon teams and businesses, from Prime de..."
1,Deloitte,4.1,23 office locations in United States,Accounting & Tax,Think a professional services career is nothin...
2,Target,3.6,4 office locations in United States,General Merchandise & Superstores,Target is one of the worlds most recognized b...
3,McDonald's,3.5,110 N Carpenter Street,Restaurants & Cafes,McDonalds is proud to be one of the most reco...
4,Accenture,4.0,United States,Business Consulting,Accenture is a global professional services co...
...,...,...,...,...,...
195,Randstad US,3.7,25 office locations in United States,HR Consulting,Randstad is the global leader in the HR servic...
196,Adecco,3.7,"10151 Deerwood Park Blvd, Bldg 400",Staffing & Subcontracting,"eBay is where the world goes to shop, sell, an..."
197,eBay,4.1,15 office locations in United States,Internet & Web Services,UST is a global digital transformations soluti...
198,UST,3.9,8 office locations in United States,Information Technology Support Services,Genpact (NYSE: G) is a global professional ser...


In [141]:
import pandas as pd

# Define bins and labels
bins = [0, 2.5, 4, 5]
labels = [1, 2, 3]

# Add the Security column
data['Security'] = pd.cut(data['Company rating'], bins=bins, labels=labels, include_lowest=True)
data

/tmp/ipykernel_32/3763719133.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['Security'] = pd.cut(data['Company rating'], bins=bins, labels=labels, include_lowest=True)


,Company Name,Company rating,Location,Industry Type,Company Description,Security
0,Amazon,3.8,25 office locations in United States,Internet & Web Services,"All Amazon teams and businesses, from Prime de...",2
1,Deloitte,4.1,23 office locations in United States,Accounting & Tax,Think a professional services career is nothin...,3
2,Target,3.6,4 office locations in United States,General Merchandise & Superstores,Target is one of the worlds most recognized b...,2
3,McDonald's,3.5,110 N Carpenter Street,Restaurants & Cafes,McDonalds is proud to be one of the most reco...,2
4,Accenture,4.0,United States,Business Consulting,Accenture is a global professional services co...,2
...,...,...,...,...,...,...
195,Randstad US,3.7,25 office locations in United States,HR Consulting,Randstad is the global leader in the HR servic...,2
196,Adecco,3.7,"10151 Deerwood Park Blvd, Bldg 400",Staffing & Subcontracting,"eBay is where the world goes to shop, sell, an...",2
197,eBay,4.1,15 office locations in United States,Internet & Web Services,UST is a global digital transformations soluti...,3
198,UST,3.9,8 office locations in United States,Information Technology Support Services,Genpact (NYSE: G) is a global professional ser...,2


In [142]:
df=data
df.columns = ['Company Name', 'Company Rating', 'Location', 'Industry Type', 'Company Description', 'Security']
df

,Company Name,Company Rating,Location,Industry Type,Company Description,Security
0,Amazon,3.8,25 office locations in United States,Internet & Web Services,"All Amazon teams and businesses, from Prime de...",2
1,Deloitte,4.1,23 office locations in United States,Accounting & Tax,Think a professional services career is nothin...,3
2,Target,3.6,4 office locations in United States,General Merchandise & Superstores,Target is one of the worlds most recognized b...,2
3,McDonald's,3.5,110 N Carpenter Street,Restaurants & Cafes,McDonalds is proud to be one of the most reco...,2
4,Accenture,4.0,United States,Business Consulting,Accenture is a global professional services co...,2
...,...,...,...,...,...,...
195,Randstad US,3.7,25 office locations in United States,HR Consulting,Randstad is the global leader in the HR servic...,2
196,Adecco,3.7,"10151 Deerwood Park Blvd, Bldg 400",Staffing & Subcontracting,"eBay is where the world goes to shop, sell, an...",2
197,eBay,4.1,15 office locations in United States,Internet & Web Services,UST is a global digital transformations soluti...,3
198,UST,3.9,8 office locations in United States,Information Technology Support Services,Genpact (NYSE: G) is a global professional ser...,2


In [143]:
from langchain.schema import Document

docs = []
for _, row in df.iterrows():
    content = f"""
Company: {row['Company Name']}
Rating: {row['Company Rating']}
Location: {row['Location']}
Industry: {row['Industry Type']}
Description: {row['Company Description']}
"""
    docs.append(
        Document(page_content=content.strip(), metadata={"security_level": int(row['Security'])})
    )


In [144]:
docs

[Document(page_content='Company: Amazon\nRating: 3.8\nLocation: 25 office locations in United States\nIndustry: Internet & Web Services\nDescription: All Amazon teams and businesses, from Prime delivery to AWS, are guided by four key tenets: customer obsession rather than competitor focus, passion for invention, commitment to operational excellence, and long-term thinking. We are driven by the excitement of building technologies, inventing products, and providing services that transform the way our customers live their lives and run their businesses.', metadata={'security_level': 2}),
 Document(page_content='Company: Deloitte\nRating: 4.1\nLocation: 23 office locations in United States\nIndustry: Accounting & Tax\nDescription: Think a professional services career is nothing but spreadsheets, gray suits, and corporate profits? Think again. From professional growth to pursuing your passions, careers at Deloitte come with plenty of opportunities. Our range of services and depth of resourc

In [145]:
from langchain.text_splitter import CharacterTextSplitter

text_splitter = CharacterTextSplitter(chunk_size=100, chunk_overlap=50)
split_docs = text_splitter.split_documents(docs)


In [146]:
model_name = "sentence-transformers/all-mpnet-base-v2"
model_kwargs = {"device": "cuda"}

embedding_model = HuggingFaceEmbeddings(model_name=model_name, model_kwargs=model_kwargs)

In [147]:
from langchain.vectorstores import Chroma

vectordb = Chroma.from_documents(
    documents=split_docs,
    embedding=embedding_model,
    persist_directory="chroma_db"  # can be saved for later use
)
vectordb.persist()


Batches:   0%|          | 0/7 [00:00<?, ?it/s]

In [148]:
def secure_retrieve(query, agent_level, top_k=2):
    retriever = vectordb.as_retriever(search_kwargs={"k": top_k})
    results = retriever.get_relevant_documents(query)
    
    filtered = [
        doc for doc in results if doc.metadata.get("security_level", 0) <= agent_level
    ]
    
    if not filtered:
        return "Access Denied — Clearance Insufficient." if results else "Oops!! No matching data found."

    return "\n\n".join([doc.page_content for doc in filtered])


In [89]:
pip install langchain --upgrade

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 33.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.3/423.3 kB 39.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 358.2/358.2 kB 37.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 443.6/443.6 kB 41.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 8.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.8/45.8 kB 6.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.5/73.5 kB 9.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.8/132.8 kB 15.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [149]:
agent_level = 2
query = "What is Amazon's industry and tell me whats in its description?"

context = secure_retrieve(query, agent_level=agent_level)

prompt = f"""Agent Level: {agent_level}
Query: {query}
Relevant Context:
{context}

Instructions: Answer the query using only the above context."""

first_pass_result=test_model(tokenizer, query_pipeline, prompt)
first_pass_result

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

/opt/conda/lib/python3.10/site-packages/transformers/pipelines/base.py:1101: UserWarning: You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
  warnings.warn(


Test inference: 2.282 sec.
Result: Agent Level: 2
Query: What is Amazon's industry?
Relevant Context:
Company: Amazon
Rating: 3.8
Location: 25 office locations in United States
Industry: Internet & Web Services
Description: All Amazon teams and businesses, from Prime delivery to AWS, are guided by four key tenets: customer obsession rather than competitor focus, passion for invention, commitment to operational excellence, and long-term thinking. We are driven by the excitement of building technologies, inventing products, and providing services that transform the way our customers live their lives and run their businesses.

Company: Adecco
Rating: 3.7
Location: 10151 Deerwood Park Blvd, Bldg 400
Industry: Staffing & Subcontracting
Description: eBay is where the world goes to shop, sell, and give. Every day, our professionals connect millions of buyers and sellers around the globe, empowering people and creating opportunity. We're on a mission to build a better, more connected form of c

In [154]:
refined_prompt = f"""
You are an intelligence assitant.

Based on the following extracted information, write a clear and concise answer to the original query.

Original Query: {query}

Extracted Information:
{first_pass_result}

Instruction:
"""

final_answer = test_model(tokenizer, query_pipeline, refined_prompt)
print("Final Answer:", final_answer)


Test inference: 2.591 sec.
Final Answer: 
You are an intelligence assitant.

Based on the following extracted information, write a clear and concise answer to the original query.

Original Query: What is Amazon's industry?

Extracted Information:
Agent Level: 2
Query: What is Amazon's industry?
Relevant Context:
Company: Amazon
Rating: 3.8
Location: 25 office locations in United States
Industry: Internet & Web Services
Description: All Amazon teams and businesses, from Prime delivery to AWS, are guided by four key tenets: customer obsession rather than competitor focus, passion for invention, commitment to operational excellence, and long-term thinking. We are driven by the excitement of building technologies, inventing products, and providing services that transform the way our customers live their lives and run their businesses.

Company: Adecco
Rating: 3.7
Location: 10151 Deerwood Park Blvd, Bldg 400
Industry: Staffing & Subcontracting
Description: eBay is where the world goes to sh

In [158]:
def test_model(tokenizer, pipeline, prompt_to_test):
    """
    Perform a query
    Return the result as a string
    """
    time_1 = time()
    sequences = pipeline(
        prompt_to_test,
        do_sample=True,
        top_k=10,
        num_return_sequences=1,
        eos_token_id=tokenizer.eos_token_id,
        max_length=2048,  # or any suitable length
    )
    time_2 = time()
    print(f"Test inference: {round(time_2 - time_1, 3)} sec.")

    results = [seq['generated_text'] for seq in sequences]
    return results[0] if results else ""

agent_level = 2
query = "What is Amazon's industry and tell me whats in its description about its 4 key tenets?"

context = secure_retrieve(query, agent_level=agent_level)

prompt = f"""Agent Level: {agent_level}
Query: {query}
Relevant Context:
{context}
"""

first_pass_result = test_model(tokenizer, query_pipeline, prompt)

refined_prompt = f"""
You are an intelligence assitant.

Based on the following extracted information, write a clear and concise answer to the original query.

Original Query: {query}

Extracted Information:
{first_pass_result}

Instruction:Write a short answer using only the information above. Be factual and do not add any assumptions.

"""

final_answer = test_model(tokenizer, query_pipeline, refined_prompt)
print("Final Answer:", final_answer)


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Test inference: 3.741 sec.
Test inference: 12.773 sec.
Final Answer: 
You are an intelligence assitant.

Based on the following extracted information, write a clear and concise answer to the original query.

Original Query: What is Amazon's industry and tell me whats in its description about its 4 key tenets?

Extracted Information:
Agent Level: 2
Query: What is Amazon's industry and tell me whats in its description about its 4 key tenets?
Relevant Context:
Company: Amazon
Rating: 3.8
Location: 25 office locations in United States
Industry: Internet & Web Services
Description: All Amazon teams and businesses, from Prime delivery to AWS, are guided by four key tenets: customer obsession rather than competitor focus, passion for invention, commitment to operational excellence, and long-term thinking. We are driven by the excitement of building technologies, inventing products, and providing services that transform the way our customers live their lives and run their businesses.

Company:

In [14]:
loader = PyMuPDFLoader("/kaggle/input/lawandorder/Bharatiya_Nyay_(Second)_Sanhita_2023.pdf")
document1 = loader.load()

## Split data in chunks

We split data in chunks using a recursive character text splitter.

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=256, chunk_overlap=20)
all_splits = text_splitter.split_documents(documents)

## Creating Embeddings and Storing in Vector Store

Create the embeddings using Sentence Transformer and HuggingFace embeddings.

In [ ]:
model_name = "sentence-transformers/all-mpnet-base-v2"
model_kwargs = {"device": "cuda"}

embeddings = HuggingFaceEmbeddings(model_name=model_name, model_kwargs=model_kwargs)

Initialize ChromaDB with the document splits, the embeddings defined previously and with the option to persist it locally.

In [ ]:
vectordb = Chroma.from_documents(documents=all_splits, embedding=embeddings, persist_directory="chroma_db")

## Initialize chain

## Test the Retrieval-Augmented Generation 


We define a test function, that will run the query and time it.

In [74]:
def test_rag(qa, query):
    print(f"Query: {query}\n")
    time_1 = time()
    result = qa.run(query)
    time_2 = time()
    print(f"Inference time: {round(time_2-time_1, 3)} sec.")
    print("\nResult: ", result)

Let's check few queries.

In [75]:
query = """
you are an intelligent assistant to assist educators to generate curricullam based question on the provideed pdf of class 8th mathematics
Instructions for Generating Mathematics Problems:-
Topic Coverage: Ensure that the problems you create span the various topics covered in this chapter.
Originality: The questions you generate should not be identical to or minor variations of the questions already present in the PDFs. Instead, use the concepts and examples as a foundation to create entirely new problems.

Now generate 2 mathematics questions.
"""
test_rag(qa, query)

NameError: name 'qa' is not defined

## Document sources

Let's check the documents sources, for the last query run.


We used Langchain, ChromaDB and Llama 2 as a LLM to build a Retrieval Augmented Generation solution. 
For IIIT DHARWAD, FOR THE TOPIC OF 'CURRICULAM BASED QUESTION GENERATION'-(FOR EXACT TOPIC AND QUESTION RELATIONAL MATCHES)




## WE SHALL CLEAR DOWN THE DATABASE MEMORY FOR NEXT BOOK READ

In [139]:
import shutil as st
st.rmtree("/kaggle/working/chroma_db")

# References  

[1] Murtuza Kazmi, Using LLaMA 2.0, FAISS and LangChain for Question-Answering on Your Own Data, https://medium.com/@murtuza753/using-llama-2-0-faiss-and-langchain-for-question-answering-on-your-own-data-682241488476  

[2] Patrick Lewis, Ethan Perez, et. al., Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks, https://browse.arxiv.org/pdf/2005.11401.pdf 

[3] Minhajul Hoque, Retrieval Augmented Generation: Grounding AI Responses in Factual Data, https://medium.com/@minh.hoque/retrieval-augmented-generation-grounding-ai-responses-in-factual-data-b7855c059322  

[4] Fangrui Liu	, Discover the Performance Gain with Retrieval Augmented Generation, https://thenewstack.io/discover-the-performance-gain-with-retrieval-augmented-generation/

[5] Andrew, How to use Retrieval-Augmented Generation (RAG) with Llama 2, https://agi-sphere.com/retrieval-augmented-generation-llama2/   

[6] Yogendra Sisodia, Retrieval Augmented Generation Using Llama2 And Falcon, https://medium.com/@scholarly360/retrieval-augmented-generation-using-llama2-and-falcon-ed26c7b14670   

